In [3]:
%pip install groq requests beautifulsoup4 langchain-text-splitters sentence-transformers faiss-cpu gradio tqdm python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import time
import json
import requests
import pickle
import numpy as np
import gradio as gr
from tqdm import tqdm
from bs4 import BeautifulSoup
from groq import Groq
from sentence_transformers import SentenceTransformer
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

# ==========================================
# 1. INITIALIZATION & CONFIGURATION
# ==========================================
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment or .env file.")

groq_client = Groq(api_key=GROQ_API_KEY)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

WIKI_BASE_URL = "https://growagarden.fandom.com"
ALL_PAGES_URL = "https://growagarden.fandom.com/wiki/Special:AllPages"

vector_db = None
document_chunks = []
chunk_metadata = []

# ==========================================
# 2. AUTOMATED WIKI CRAWLER & SCRAPER
# ==========================================
def get_all_wiki_titles(base_url=WIKI_BASE_URL):
    """Fetches ALL article titles using the MediaWiki API."""
    api_url = f"{base_url.rstrip('/')}/api.php"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    wiki_titles = []
    apcontinue = None

    print(f"🔍 Fetching page list from API...")
    while True:
        params = {
            "action": "query",
            "list": "allpages",
            "aplimit": "max",
            "format": "json"
        }
        if apcontinue:
            params["apcontinue"] = apcontinue

        try:
            response = requests.get(api_url, headers=headers, params=params, timeout=10)
            if response.status_code != 200:
                break
            data = response.json()
            pages = data.get("query", {}).get("allpages", [])
            for p in pages:
                title = p["title"]
                # Filter out system pages, files, and templates
                if not any(title.startswith(prefix) for prefix in ["File:", "Category:", "Talk:", "User:", "Template:", "MediaWiki:"]):
                    wiki_titles.append(title)
            if "continue" in data and "apcontinue" in data["continue"]:
                apcontinue = data["continue"]["apcontinue"]
            else:
                break
        except Exception:
            break

    return wiki_titles


def scrape_single_page_via_api(title, base_url=WIKI_BASE_URL):
    """Fetches clean article text directly through the MediaWiki Parse API."""
    api_url = f"{base_url.rstrip('/')}/api.php"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    params = {
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json"
    }

    try:
        response = requests.get(api_url, headers=headers, params=params, timeout=10)
        if response.status_code != 200:
            return None, None

        data = response.json()
        if "parse" not in data or "text" not in data["parse"]:
            return None, None

        raw_html = data["parse"]["text"]["*"]
        soup = BeautifulSoup(raw_html, 'html.parser')

        # Clean scripts and styles, but keep tables and infoboxes intact!
        for element in soup.find_all(['script', 'style', 'noscript']):
            element.decompose()

        # Extract all readable text
        text = soup.get_text(separator=' ')
        
        # Clean formatting, citation brackets, and duplicate spaces
        clean_text = re.sub(r'\[\d+\]', '', text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()

        return title, clean_text
    except Exception:
        return None, None


def crawl_and_build_database(max_pages=150):
    """Crawls titles via API, chunks text, embeds, and indexes into FAISS."""
    global vector_db, document_chunks, chunk_metadata

    titles = get_all_wiki_titles(WIKI_BASE_URL)
    if not titles:
        return "❌ Could not retrieve titles from MediaWiki API."

    titles = titles[:max_pages]  # Limit total pages processed
    document_chunks = []
    chunk_metadata = []
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

    scraped_count = 0
    print(f"🚀 Processing {len(titles)} wiki pages directly via API...")

    for title in tqdm(titles, desc="Parsing Articles"):
        page_title, text = scrape_single_page_via_api(title, WIKI_BASE_URL)
        if text and len(text) > 50:
            chunks = text_splitter.split_text(text)
            for chunk in chunks:
                document_chunks.append(chunk)
                chunk_metadata.append({
                    "title": page_title, 
                    "url": f"{WIKI_BASE_URL}/wiki/{page_title.replace(' ', '_')}"
                })
            scraped_count += 1
        time.sleep(0.05)  # Fast API requests

    if not document_chunks:
        return "❌ Failed to extract content from wiki pages."

    # Build FAISS Vector Index
    print("\n🧠 Generating embeddings and creating FAISS index...")
    embeddings = embedding_model.encode(document_chunks, show_progress_bar=True)
    embedding_dim = embeddings.shape[1]

    vector_db = faiss.IndexFlatL2(embedding_dim)
    vector_db.add(np.array(embeddings).astype('float32'))

    # Save to disk for quick reuse
    faiss.write_index(vector_db, "grow_a_garden.index")
    with open("knowledge_data.pkl", "wb") as f:
        pickle.dump({"chunks": document_chunks, "metadata": chunk_metadata}, f)

    return f"✅ Successfully indexed {scraped_count} pages! Created {len(document_chunks)} total knowledge chunks in FAISS."

def load_existing_database():
    """Loads pre-built vector DB from local files if available."""
    global vector_db, document_chunks, chunk_metadata

    if os.path.exists("grow_a_garden.index") and os.path.exists("knowledge_data.pkl"):
        vector_db = faiss.read_index("grow_a_garden.index")
        with open("knowledge_data.pkl", "rb") as f:
            data = pickle.load(f)
            document_chunks = data["chunks"]
            chunk_metadata = data["metadata"]
        return f"⚡ Loaded existing vector database with {len(document_chunks)} chunks!"
    return "ℹ️ No existing database found. Click 'Run Auto-Crawler' to build it."

# ==========================================
# 3. RETRIEVAL & GROQ GENERATION PIPELINE
# ==========================================
def answer_game_query(user_question, system_personality):
    global vector_db, document_chunks, chunk_metadata

    if vector_db is None or len(document_chunks) == 0:
        return "⚠️ Please run or load the Auto-Crawler database first!", "No sources loaded."

    # Embed query and perform similarity search
    query_vector = embedding_model.encode([user_question]).astype('float32')
    distances, indices = vector_db.search(query_vector, k=4)

    retrieved_chunks = []
    citations = []

    for idx in indices[0]:
        if idx < len(document_chunks):
            retrieved_chunks.append(document_chunks[idx])
            meta = chunk_metadata[idx]
            citations.append(f"📌 **[{meta['title']}]({meta['url']})**\n> {document_chunks[idx]}")

    context_str = "\n\n---\n\n".join(retrieved_chunks)

    # Prompt Template
    rag_prompt = f"""You are an expert game assistant for the game 'Grow a Garden'. 
Use the provided context pulled directly from the official Fandom Wiki to answer the user's query accurately.
If the information is not present in the context, clearly state that you don't know based on current wiki data.

WIKI CONTEXT:
{context_str}

PLAYER QUESTION:
{user_question}
"""

    completion = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_personality},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.5
    )

    answer = completion.choices[0].message.content
    sources_text = "\n\n".join(citations)

    return answer, sources_text

# ==========================================
# 4. GRADIO INTERFACE
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🪴 Grow a Garden - Auto-Crawled Wiki AI Assistant")
    gr.Markdown("Full RAG Chatbot powered by Groq, FAISS, SentenceTransformers, and automated web crawling.")

    with gr.Tab("1. Knowledge Base Manager"):
        gr.Markdown("### Crawl and Index the Wiki")
        crawl_btn = gr.Button("🌐 Run Auto-Crawler (Scrape All Wiki Pages)")
        load_btn = gr.Button("⚡ Load Existing Index from Disk")
        status_box = gr.Textbox(label="Database Status", value=load_existing_database())

        crawl_btn.click(fn=crawl_and_build_database, outputs=[status_box])
        load_btn.click(fn=load_existing_database, outputs=[status_box])

    with gr.Tab("2. Query Assistant"):
        personality_input = gr.Textbox(
            label="System Personality Prompt", 
            value="You are a helpful and detailed game wiki expert for 'Grow a Garden'."
        )
        question_input = gr.Textbox(
            label="Ask a Question about Grow a Garden", 
            placeholder="What does the Gourmet Egg hatch into, or how do I get a Bagel Bunny?"
        )
        submit_btn = gr.Button("Search Wiki & Answer")

        answer_output = gr.Textbox(label="Chatbot Answer", lines=6)
        sources_output = gr.Markdown(label="Wiki Sources & Citations")

        submit_btn.click(
            fn=answer_game_query, 
            inputs=[question_input, personality_input], 
            outputs=[answer_output, sources_output]
        )

# Launch app
demo.launch(share=True, debug=True)

c:\Coding Projects\Non-creditAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2844.78it/s]
C:\Users\anush\AppData\Local\Temp\ipykernel_24516\2963560182.py:225: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://87fff3a3ead3e0f4ce.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://87fff3a3ead3e0f4ce.gradio.live
